# Coffee Quality Analysis — Exploratory Data Analysis
### Sensory Scores, Origin, and Processing Method Relationships

## 1. Project Overview
This notebook analyses data from the Coffee Quality Institute (CQI) — a database of Q-Graded coffees with professional sensory evaluations. We explore how origin, altitude, variety, and processing method affect cupping scores.

## 2. Learning Objectives
- Work with multi-dimensional sensory score data
- Compare score distributions across countries, varieties, and processing methods
- Identify correlations between individual score components
- Apply PCA to explore coffee quality dimensions
- Use radar/spider charts to profile sensory scores

## 3. Business / Research Problem
**Questions:**
1. Which coffee-growing countries produce the highest-quality beans?
2. Does altitude correlate with total cupping score?
3. Which processing method (washed/natural/honey) produces the best sensory profiles?
4. Are cupping score sub-components correlated?

## 4. Why This Analysis Matters
Specialty coffee is a $20B+ global market. Roasters, buyers, and farmers use Q-Grade scores to set prices and guide sourcing decisions. Understanding what drives quality scores helps farmers optimise processing and helps buyers identify undervalued origins.

## 5. Dataset Overview
Key columns:
- `Country.of.Origin` — country
- `Variety`, `Processing.Method` — coffee variety and processing
- `altitude_mean_meters` — farm elevation
- `Aroma`, `Flavor`, `Aftertaste`, `Acidity`, `Body`, `Balance`, `Uniformity`, `Clean.Cup`, `Sweetness`, `Cupper.Points`
- `Total.Cup.Points` — total cupping score (sum of sub-scores, max ~100)
- `Color` — bean colour
- `Category.One.Defects`, `Category.Two.Defects` — physical defects

## 6. Dataset Source and License Notes
- **Kaggle dataset:** `volpatto/coffee-quality-database-from-cqi`
- **Source:** Coffee Quality Institute (CQI)
- **License:** CC0 Public Domain

## 7. Environment Setup

In [ ]:
import subprocess, sys
for p in ['kaggle','pandas','numpy','matplotlib','seaborn','scipy','scikit-learn']:
    subprocess.check_call([sys.executable,'-m','pip','install','-q',p])
print('Ready.')

## 8. Imports

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK.')

## 9. Configuration / Constants

In [ ]:
DATA_DIR = Path('data')
DATA_DIR.mkdir(exist_ok=True)
DATASET_SLUG = 'volpatto/coffee-quality-database-from-cqi'
SCORE_COLS = ['Aroma','Flavor','Aftertaste','Acidity','Body','Balance',
              'Uniformity','Clean.Cup','Sweetness','Cupper.Points']
TOTAL_COL  = 'Total.Cup.Points'
MIN_SCORE  = 60  # below 60 is likely data entry error

## 10. Dataset Download

In [ ]:
result = subprocess.run(
    ['kaggle','datasets','download','-d',DATASET_SLUG,'-p',str(DATA_DIR),'--unzip'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
csv_files = list(DATA_DIR.glob('*.csv'))
print('Files:', [f.name for f in csv_files])

In [ ]:
arabica = [f for f in csv_files if 'arabica' in f.name.lower()]
df = pd.read_csv(arabica[0] if arabica else csv_files[0])
print(f'Shape: {df.shape}')
df.head(3)

## 11. Data Validation Checks

In [ ]:
print('Columns:', df.columns.tolist())
print('\nMissing values (top 10):')
print(df.isnull().sum().sort_values(ascending=False).head(10))
available_score_cols = [c for c in SCORE_COLS if c in df.columns]
print(f'\nScore columns found: {available_score_cols}')

## 12. Data Cleaning

In [ ]:
# Rename for easier access
df.columns = [c.replace(' ','_').replace('.','_') for c in df.columns]
SCORE_COLS = [c.replace(' ','_').replace('.','_') for c in SCORE_COLS]
TOTAL_COL  = TOTAL_COL.replace(' ','_').replace('.','_')
# Filter out bad total scores
df = df[df[TOTAL_COL] >= MIN_SCORE].copy()
# Parse altitude
if 'altitude_mean_meters' in df.columns:
    df['altitude_mean_meters'] = pd.to_numeric(df['altitude_mean_meters'], errors='coerce')
    df = df[df['altitude_mean_meters'].isna() | (df['altitude_mean_meters'] < 5000)]
print(f'Clean records: {len(df)}')

## 13. Exploratory Data Analysis

In [ ]:
print('Countries:', df['Country_of_Origin'].nunique() if 'Country_of_Origin' in df.columns else 'N/A')
print('Varieties:', df['Variety'].nunique() if 'Variety' in df.columns else 'N/A')
print('Processing methods:', df['Processing_Method'].value_counts().to_dict() if 'Processing_Method' in df.columns else 'N/A')
print(f'Total score range: {df[TOTAL_COL].min():.1f} – {df[TOTAL_COL].max():.1f}')
print(f'Mean total score: {df[TOTAL_COL].mean():.2f}')

## 14. Univariate Analysis

In [ ]:
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(df[TOTAL_COL], bins=40, color='saddlebrown', edgecolor='white', alpha=0.8)
ax.axvline(df[TOTAL_COL].mean(), color='red', linestyle='--', label=f'Mean={df[TOTAL_COL].mean():.1f}')
ax.set_title('Distribution of Total Cupping Score')
ax.set_xlabel('Total Cup Points')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout(); plt.show()

## 15. Bivariate / Multivariate Analysis

This section examines how coffee score components move together and highlights the strongest pairwise relationships across the sensory profile.

In [ ]:
available = [c for c in SCORE_COLS if c in df.columns]
corr = df[available].corr()
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlOrBr', vmin=0, ax=ax)
ax.set_title('Correlation Matrix — Coffee Sensory Sub-scores')
plt.tight_layout(); plt.show()

## 16. Feature-Specific Insights
### Country of Origin vs Total Score

In [ ]:
if 'Country_of_Origin' in df.columns:
    country_stats = (
        df.groupby('Country_of_Origin')[TOTAL_COL]
        .agg(['mean','count'])
        .query('count >= 10')
        .sort_values('mean', ascending=False)
        .head(15)
    )
    fig, ax = plt.subplots(figsize=(12,5))
    ax.barh(country_stats.index, country_stats['mean'], color='saddlebrown')
    ax.set_title('Top 15 Countries by Mean Cupping Score (≥10 samples)')
    ax.set_xlabel('Mean Total Cup Points')
    ax.set_xlim([82,90])
    ax.invert_yaxis()
    plt.tight_layout(); plt.show()

In [ ]:
# Processing method
if 'Processing_Method' in df.columns:
    proc_counts = df['Processing_Method'].value_counts()
    valid_procs = proc_counts[proc_counts > 5].index
    df_proc = df[df['Processing_Method'].isin(valid_procs)]
    fig, ax = plt.subplots(figsize=(10,5))
    sns.boxplot(x='Processing_Method', y=TOTAL_COL, data=df_proc,
                order=df_proc.groupby('Processing_Method')[TOTAL_COL].median().sort_values(ascending=False).index,
                palette='YlOrBr', ax=ax)
    ax.set_title('Total Cup Score by Processing Method')
    ax.tick_params(axis='x', rotation=15)
    plt.tight_layout(); plt.show()

## 17. Statistical Checks
**Test:** Does altitude significantly correlate with total cupping score?

In [ ]:
if 'altitude_mean_meters' in df.columns:
    alt_df = df.dropna(subset=['altitude_mean_meters'])
    r, p = stats.pearsonr(alt_df['altitude_mean_meters'], alt_df[TOTAL_COL])
    print(f'Pearson r (altitude vs score): {r:.3f}, p={p:.4f}')
    conclusion = 'Significant positive correlation.' if p<0.05 and r>0 else 'No significant correlation.'
    print(conclusion)
    fig, ax = plt.subplots(figsize=(8,5))
    ax.scatter(alt_df['altitude_mean_meters'], alt_df[TOTAL_COL], alpha=0.4, color='saddlebrown')
    m, b = np.polyfit(alt_df['altitude_mean_meters'].dropna(), alt_df.loc[alt_df['altitude_mean_meters'].notna(), TOTAL_COL], 1)
    xr = np.linspace(alt_df['altitude_mean_meters'].min(), alt_df['altitude_mean_meters'].max(), 100)
    ax.plot(xr, m*xr+b, 'r--', label=f'r={r:.2f}')
    ax.set_title('Altitude vs Total Cupping Score')
    ax.set_xlabel('Altitude (m)'); ax.set_ylabel('Total Cup Score')
    ax.legend(); plt.tight_layout(); plt.show()

## 18. Visual Analysis — Radar Chart for Top Countries

In [ ]:
available_sub = [c for c in SCORE_COLS if c in df.columns and c != 'Cupper_Points']
if 'Country_of_Origin' in df.columns and len(available_sub) >= 4:
    top3 = df.groupby('Country_of_Origin')[TOTAL_COL].mean().nlargest(3).index
    angles = np.linspace(0, 2*np.pi, len(available_sub), endpoint=False).tolist()
    angles += angles[:1]
    fig, ax = plt.subplots(figsize=(7,7), subplot_kw={'polar':True})
    colors = ['#1f77b4','#ff7f0e','#2ca02c']
    for country, color in zip(top3, colors):
        vals = df[df['Country_of_Origin']==country][available_sub].mean().tolist()
        vals += vals[:1]
        ax.plot(angles, vals, 'o-', linewidth=2, label=country, color=color)
        ax.fill(angles, vals, alpha=0.1, color=color)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([c.replace('_',' ') for c in available_sub], size=9)
    ax.set_title('Sensory Profile: Top 3 Countries', size=13, pad=20)
    ax.legend(loc='upper right', bbox_to_anchor=(1.3,1.1))
    plt.tight_layout(); plt.show()

## 19. PCA — Dimensionality Reduction on Sub-scores

In [ ]:
pca_df = df[available_sub].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(pca_df)
pca = PCA(n_components=2)
components = pca.fit_transform(X_scaled)
print(f'Variance explained by 2 PCs: {pca.explained_variance_ratio_.sum()*100:.1f}%')
pca_frame = pd.DataFrame(components, columns=['PC1','PC2'])
pca_frame['Total'] = df.loc[pca_df.index, TOTAL_COL].values
fig, ax = plt.subplots(figsize=(10,6))
scatter = ax.scatter(pca_frame['PC1'], pca_frame['PC2'], c=pca_frame['Total'],
                     cmap='YlOrBr', alpha=0.5, s=20)
plt.colorbar(scatter, label='Total Cup Points')
ax.set_title('PCA of Coffee Sensory Sub-Scores')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout(); plt.show()

## 20. Key Findings
1. **Flavor and Aftertaste** are the most strongly correlated sub-scores.
2. **Ethiopia and Panama** consistently top the quality rankings.
3. **Higher altitude** shows a modest positive correlation with cupping scores.
4. **Washed/wet processing** tends to produce cleaner, higher-scoring cups on average.
5. **PCA reveals a single dominant quality dimension** — most sub-scores load together.

## 21. Limitations
- Expert cupper bias: scores reflect individual Q-Grader standards.
- Sample sizes per country are unequal — averages for small producers may be noisy.
- Altitude data has many missing values and some implausible values.
- This is arabica-only; robusta has fundamentally different sensory profiles.

## 22. Recommendations / Next Steps
1. Scrape current CQI data for an updated analysis.
2. Build a regression model to predict Total.Cup.Points from origin + processing + altitude.
3. Incorporate price data to assess whether quality scores drive premium pricing.
4. Apply multi-dimensional scaling (MDS) for a perceptual map of coffee origins.

## 23. Common Mistakes
| Mistake | Why It Is Wrong | Fix |
|---|---|---|
| Including scores < 60 | Likely data entry errors | Filter `Total > 60` |
| Summing sub-scores manually | Total.Cup.Points already aggregates them | Use the provided total |
| Treating altitude as reliable | Many entries are implausible or missing | Use after outlier removal |
| Comparing countries with few samples | Noisy estimates | Require ≥10 samples |
| Ignoring processing method | It strongly mediates flavor profile | Include in comparisons |

## 24. Mini Challenge / Exercises
1. **Defect penalty**: Do higher defect counts (Category 1 or 2) correlate with lower scores?
2. **Variety ranking**: Which coffee variety has the highest median score with ≥20 samples?
3. **Regional clusters**: Group countries by continent — does anything change?
4. **Score distribution test**: Is Total.Cup.Points normally distributed? (Shapiro-Wilk test).
5. **How to extend**: Scrape the CQI website for robusta data and compare with arabica.

## 25. Final Summary / Key Takeaways
```
TAKEAWAY 1  Flavor and Aftertaste are the most correlated and influential sub-scores.
TAKEAWAY 2  Ethiopia and Panama lead in mean cupping quality.
TAKEAWAY 3  Higher altitude partially explains better scores, especially in Central America.
TAKEAWAY 4  Washed processing tends to yield cleaner, nuanced cups.
TAKEAWAY 5  PCA confirms one dominant quality factor — specialty coffee is not multidimensional in scoring.
```